<h1>-----------📰FAKE AND REAL NEWS DETECTION USING RNN----------</h1>

## TABLE OF CONTENTS:
1.Project Objective

2.Problem Statement

3.Import Libraries

4.Load Dataset

5.Understand the Dataset

6.Create Target Variable

7.Combine Title and New Text

8.Exploratory Data Analysis

9.Data Cleaning

10.Remove Empty 

11.Separate Features and Target

12.Train Test 

13.Tokenization

14.Padding

15.Build RNN 

16.Compile Model

17.Train RNN

18.Evaluate the Model

19.Prediction

20.Classification Report

21.Confusion Matrix

22.Predict New and Unseen News

23.Conclusion

## Project Objective:

1.With the increasing number of Fraud website on the internet which increases the number of fake news day by day, people are profiting by clickbaits and publishing fake news on online. By clicking on a clickbait, users are led to a page that contains false information. More clicks contributes to more money for content publishers and fake news influences people perceptions.

2.The rise of Fake news has become a global problem that even major tech companies like Facebook and Google are struggling to solve. It can be difficult to determine whether a text is factual without additional context and human judgement.



## Project Statement:

The given news dataset consists of two folders consisting of True and Fake news articles. The objective is to develop a robust model that can accurately identify and classify a given news as Real or Fake. The project will involve the following steps:

1.	Data Preparation: Preprocessing the dataset by Cleaning Data, Bag of Words, Stemming, Lemmatization, Tokenization, MultinomialNB Algorithm, TF-IDF, word vectorization, word2 vec and splitting the dataset into training and testing sets.
2.	Model Building: Developing a Recurrent Neural Network (RNN) model from scratch that can accurately classify the text as Real or Fake news. 
3.	Model Evaluation: Evaluating the model's performance on the testing set to measure its accuracy and other performance metrics such as Precision, Recall, and F1-score. The model's generalization ability will also be evaluated by testing it on new and unseen news.


<h1>IMPORT LIBRARIES</h1>

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

<h1>LOAD DATASET</h1>

In [2]:
fake = pd.read_csv(r"C:\Users\reena\Downloads\Fake.csv")
true = pd.read_csv(r"C:\Users\reena\Downloads\True.csv")

print("Fake:", fake.shape)
print("True:", true.shape)

Fake: (23481, 4)
True: (21417, 4)


<h1>UNDERSTAND THE DATASET</h1>

In [3]:
print(fake.head())

                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  


In [4]:
print(true.head())

                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   
1  U.S. military to accept transgender recruits o...   
2  Senior U.S. Republican senator: 'Let Mr. Muell...   
3  FBI Russia probe helped by Australian diplomat...   
4  Trump wants Postal Service to charge 'much mor...   

                                                text       subject  \
0  WASHINGTON (Reuters) - The head of a conservat...  politicsNews   
1  WASHINGTON (Reuters) - Transgender people will...  politicsNews   
2  WASHINGTON (Reuters) - The special counsel inv...  politicsNews   
3  WASHINGTON (Reuters) - Trump campaign adviser ...  politicsNews   
4  SEATTLE/WASHINGTON (Reuters) - President Donal...  politicsNews   

                 date  
0  December 31, 2017   
1  December 29, 2017   
2  December 31, 2017   
3  December 30, 2017   
4  December 29, 2017   


In [5]:
print(fake.columns)

Index(['title', 'text', 'subject', 'date'], dtype='object')


In [6]:
print(fake.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB
None


In [7]:
#checking missing values

print(fake.isnull().sum())
print(true.isnull().sum())

title      0
text       0
subject    0
date       0
dtype: int64
title      0
text       0
subject    0
date       0
dtype: int64


<h1>CREATE TARGET VARIABLE</h1>

In [8]:
#Here fake-0,label=1
fake["label"] = 0
true["label"] = 1

In [9]:
#combine both datasets

df = pd.concat([fake, true], ignore_index=True)

print(df.shape)

(44898, 5)


In [10]:
#Check class distibution

print(df["label"].value_counts())

label
0    23481
1    21417
Name: count, dtype: int64


<h1>COMBINE TITLE AND NEWS TEXT</h1>

In [11]:
df["content"] = (
    df["title"].fillna("") + " " +
    df["text"].fillna("")
)

In [12]:
print(df["content"].head())

0     Donald Trump Sends Out Embarrassing New Year’...
1     Drunk Bragging Trump Staffer Started Russian ...
2     Sheriff David Clarke Becomes An Internet Joke...
3     Trump Is So Obsessed He Even Has Obama’s Name...
4     Pope Francis Just Called Out Donald Trump Dur...
Name: content, dtype: object


<h1>EXPLORATORY DATA ANALYSIS</h1>

<h1>DATA CLEANING</h1>

In [13]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [14]:
df["clean_text"] = df["content"].apply(clean_text)

In [15]:
print(df[["content", "clean_text"]].head())

                                             content  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                          clean_text  
0  donald trump sends out embarrassing new years ...  
1  drunk bragging trump staffer started russian c...  
2  sheriff david clarke becomes an internet joke ...  
3  trump is so obsessed he even has obamas name c...  
4  pope francis just called out donald trump duri...  


<h1>REMOVE EMPTY TEXT</h1>

In [17]:
df = df[df["clean_text"].str.strip() != ""]

In [18]:
print(df.shape)

(44889, 7)


<h1>SEPARATE FEATURES AND TARGET</h1>

In [19]:
X = df["clean_text"]
y = df["label"]

<h1>TRAIN TEST SPLIT</h1>

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (35911,)
Testing data: (8978,)


<h1>TOKENZIATION</h1>

In [21]:
#RNN does not read the words directly so we have to convert words into numbers

tokenizer = Tokenizer(
    num_words=15000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

In [22]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [23]:
print(X_train_seq[0][:20])

[68, 197, 1226, 3, 191, 265, 757, 847, 110, 69, 565, 265, 225, 197, 14, 9, 200, 33, 37, 847]


<h1>PADDING</h1>

In [24]:
# News article have different length so we have to make all the sequences in the same length

max_length = 150

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [25]:
print("Training shape:", X_train_pad.shape)
print("Testing shape:", X_test_pad.shape)

Training shape: (35911, 150)
Testing shape: (8978, 150)


<h1>BUILD RNN MODEL</h1>

In [26]:
#Build Embedding-simple RNN- Dense layers
vocab_size = min(
    15000,
    len(tokenizer.word_index) + 1
)

model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=64
    )
)

model.add(SimpleRNN(64))

model.add(Dropout(0.3))

model.add(Dense(32, activation="relu"))

model.add(Dense(1, activation="sigmoid"))

<h1>COMPILE MODEL</h1>

In [27]:
#Use Adam complier and binary cross entropy

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [28]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

<h1>TRAIN RNN</h1>

In [29]:
#Train RNN using the training data
history = model.fit(
    X_train_pad,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/3
449/449 ━━━━━━━━━━━━━━━━━━━━ 39s 76ms/step - accuracy: 0.5982 - loss: 0.6548 - val_accuracy: 0.7380 - val_loss: 0.5607
Epoch 2/3
449/449 ━━━━━━━━━━━━━━━━━━━━ 36s 80ms/step - accuracy: 0.8599 - loss: 0.3439 - val_accuracy: 0.9451 - val_loss: 0.1432
Epoch 3/3
449/449 ━━━━━━━━━━━━━━━━━━━━ 43s 95ms/step - accuracy: 0.9708 - loss: 0.0852 - val_accuracy: 0.9781 - val_loss: 0.0745


<h1>EVALUATE THE MODEL</h1>

In [30]:
#Calculate the accuary and loss
loss, accuracy = model.evaluate(
    X_test_pad,
    y_test
)

print("Test Accuracy:", accuracy)

281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9758 - loss: 0.0861
Test Accuracy: 0.9758297801017761


In [31]:
print("Test Accuracy:", accuracy * 100, "%")

Test Accuracy: 97.58297801017761 %


<h1>PREDICTION</h1>

In [32]:
y_prob = model.predict(X_test_pad)

281/281 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step


In [33]:
y_pred = (y_prob >= 0.5).astype(int).ravel()

<h1>CLASSIFICATION REPORT</h1>

In [34]:
#Calculate precision,recall,f1score
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Fake", "Real"]
    )
)

              precision    recall  f1-score   support

        Fake       1.00      0.96      0.98      4695
        Real       0.95      1.00      0.98      4283

    accuracy                           0.98      8978
   macro avg       0.98      0.98      0.98      8978
weighted avg       0.98      0.98      0.98      8978



<h1> CONFUSION MATRIX</h1>

In [35]:
#Analyse correct and incorrect calculations
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[4484  211]
 [   6 4277]]


<h1>PREDICT NEW AND UNSEEN NEWS</h1>

In [36]:
#Give new news article and predict real or fake
def predict_news(news):

    # Cleaning
    news = news.lower()
    news = re.sub(r"http\S+|www\S+", "", news)
    news = re.sub(r"[^a-zA-Z\s]", "", news)
    news = re.sub(r"\s+", " ", news).strip()

    # Tokenization
    sequence = tokenizer.texts_to_sequences([news])

    # Padding
    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )

    # Prediction
    probability = model.predict(padded, verbose=0)[0][0]

    if probability >= 0.5:
        return "Real News"
    else:
        return "Fake News"

In [37]:

news = """
The government announced a new policy to improve public transportation.
"""

print(predict_news(news))

Real News


## Conclusion:

In this project, we developed an RNN-based deep learning model to classify news articles as Fake or Real. The text data was preprocessed and converted into numerical sequences using tokenization and padding. The RNN model learned patterns from the news content and successfully classified the articles into two categories. The model achieved good validation performance, showing that RNN can be effectively used for text classification and fake news detection. This project helped us understand NLP, text preprocessing, sequence modeling, and deep learning.